In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from weather_client import WeatherClient

client = WeatherClient()

location = "Chicago, IL"
lat = 41.8781
lon = -87.6298

In [0]:
%pip install -U sentence-transformers

In [0]:
point = client.get_point_metadata(lat, lon)

print(point["properties"]["forecast"])

In [0]:
forecasts = client.get_forecast(lat, lon)

print("Forecast periods:", len(forecasts))

for period in forecasts[:2]:
    print("Name:", period.get("name"))
    print("Forecast:", period.get("detailedForecast"))
    print()

In [0]:
alerts = client.get_alerts(lat, lon)

print("Active alerts:", len(alerts))

In [0]:
documents = client.normalize_forecasts(
    location,
    lat,
    lon,
    forecasts,
)

print("Normalized documents:", len(documents))

for doc in documents[:2]:
    print("\nID:", doc["id"])
    print("Location:", doc["location"])
    print("Type:", doc["source_type"])
    print("Headline:", doc["headline"])
    print("Effective:", doc["effective_at"])
    print("Text:", doc["narrative_text"][:300])

In [0]:
alert_documents = client.normalize_alerts(
    location,
    lat,
    lon,
    alerts,
)

print("Normalized alerts:", len(alert_documents))

In [0]:
import lakebase

processed = lakebase.upsert_weather_documents(
    documents + alert_documents
)

print("Processed weather documents:", processed)

In [0]:
import lakebase

processed = lakebase.upsert_weather_documents(
    documents
)

print("Processed weather documents:", processed)

In [0]:
import sys
sys.path.append('/Workspace/Users/sainikhil123.sb@gmail.com/weather-intelligence-lakebase')
from app import app

test_client = app.test_client()

response = test_client.post(
    "/weather/sync",
    json={
        "location": "Chicago, IL",
        "latitude": 41.8781,
        "longitude": -87.6298,
    },
)

print("Status:", response.status_code)
print(response.get_json())

In [0]:
import sys
import importlib.util

module_path = '/Workspace/Users/sainikhil123.sb@gmail.com/weather-intelligence-lakebase/weather_embeddings.py'
spec = importlib.util.spec_from_file_location('weather_embeddings', module_path)
weather_embeddings = importlib.util.module_from_spec(spec)
spec.loader.exec_module(weather_embeddings)
chunk_text = weather_embeddings.chunk_text

text = documents[0]["narrative_text"]

chunks = chunk_text(text)

print("Original length:", len(text))
print("Number of chunks:", len(chunks))

for index, chunk in enumerate(chunks):
    print()
    print("Chunk:", index)
    print("Length:", len(chunk))
    print(chunk)

In [0]:
long_text = "Heavy rain and flash flooding are possible. " * 100

chunks = chunk_text(long_text)

print("Number of chunks:", len(chunks))

for index, chunk in enumerate(chunks[:3]):
    print(
        index,
        "length =",
        len(chunk),
    )

In [0]:
# from weather_embeddings import generate_embedding
module_path = '/Workspace/Users/sainikhil123.sb@gmail.com/weather-intelligence-lakebase/weather_embeddings.py'
spec = importlib.util.spec_from_file_location('weather_embeddings', module_path)
weather_embeddings = importlib.util.module_from_spec(spec)
spec.loader.exec_module(weather_embeddings)

generate_embedding = weather_embeddings.generate_embedding


test_text = (
    "Heavy rain and flash flooding "
    "are possible this weekend."
)

embedding = generate_embedding(test_text)

print("Type:", type(embedding))
print("Dimensions:", len(embedding))
print("First 5 values:", embedding[:5])

In [0]:
import sys
import importlib.util

module_path = '/Workspace/Users/sainikhil123.sb@gmail.com/weather-intelligence-lakebase/weather_embeddings.py'
spec = importlib.util.spec_from_file_location('weather_embeddings', module_path)
weather_embeddings = importlib.util.module_from_spec(spec)
spec.loader.exec_module(weather_embeddings)

chunk_text = weather_embeddings.chunk_text
generate_embedding = weather_embeddings.generate_embedding
MODEL_NAME = weather_embeddings.MODEL_NAME

import lakebase

In [0]:
document = documents[0]

chunks = chunk_text(
    document["narrative_text"]
)

chunk = chunks[0]

embedding = generate_embedding(chunk)

print("Document:", document["headline"])
print("Chunk length:", len(chunk))
print("Embedding dimensions:", len(embedding))

In [0]:
stored_documents = lakebase.run_query("""
    SELECT
        id,
        headline,
        narrative_text
    FROM weather_documents
    ORDER BY effective_at
    LIMIT 1
""")

document = stored_documents[0]

print("Document ID:", document["id"])
print("Headline:", document["headline"])
print("Text:", document["narrative_text"][:200])


chunks = chunk_text(
    document["narrative_text"]
)

chunk = chunks[0]

embedding = generate_embedding(chunk)

print("Chunk length:", len(chunk))
print("Embedding dimensions:", len(embedding))


embedding_id = (
    f"{document['id']}_chunk_0"
)

result = lakebase.upsert_weather_embedding(
    embedding_id=embedding_id,
    document_id=document["id"],
    chunk_index=0,
    chunk_text=chunk,
    embedding=embedding,
    model_name=MODEL_NAME,
)

print("Rows affected:", result)

In [0]:
import lakebase

stored_documents = lakebase.run_query("""
    SELECT
        id,
        narrative_text
    FROM weather_documents
    ORDER BY effective_at
""")

print("Stored documents:", len(stored_documents))

In [0]:
from weather_embeddings import (
    build_embeddings_for_documents,
)

embedding_records = build_embeddings_for_documents(
    stored_documents
)

print(
    "Embedding records:",
    len(embedding_records),
)

print(
    "First vector dimensions:",
    len(embedding_records[0]["embedding"]),
)

In [0]:
import importlib
import lakebase
importlib.reload(lakebase)
from lakebase import upsert_weather_embeddings

processed = upsert_weather_embeddings(
    embedding_records
)

print("Processed embeddings:", processed)

In [0]:
from weather_embeddings import generate_embedding

query = "heavy rain and flash flooding"

query_embedding = generate_embedding(query)

print("Query:", query)
print("Dimensions:", len(query_embedding))

In [0]:
query_vector = (
    "["
    + ",".join(
        str(value)
        for value in query_embedding
    )
    + "]"
)

In [0]:
results = lakebase.run_query(
    """
    SELECT
        d.location,
        d.headline,
        d.source_type,
        e.chunk_index,
        e.chunk_text,
        1 - (
            e.embedding <=> %s::vector
        ) AS similarity
    FROM weather_embeddings e
    JOIN weather_documents d
        ON d.id = e.document_id
    ORDER BY
        e.embedding <=> %s::vector
    LIMIT 5
    """,
    (
        query_vector,
        query_vector,
    ),
)

In [0]:
for row in results:
    print()
    print("Location:", row["location"])
    print("Headline:", row["headline"])
    print("Type:", row["source_type"])
    print(
        "Similarity:",
        round(float(row["similarity"]), 4)
    )
    print("Text:", row["chunk_text"])

In [0]:
%pip install flask

In [0]:
import sys
sys.path.append('/Workspace/Users/sainikhil123.sb@gmail.com/weather-intelligence-lakebase')
from app import app

test_client = app.test_client()

response = test_client.post(
    "/weather/search",
    json={
        "query": "heavy rain and flash flooding"
    },
)

print("Status:", response.status_code)

data = response.get_json()

if data is None:
    print("Error: Response did not contain JSON data")
    print("Response text:", response.get_data(as_text=True))
else:
    print("Query:", data["query"])
    print("Count:", data["count"])

    for result in data["results"]:
        print()
        print("Headline:", result["headline"])
        print(
            "Similarity:",
            round(result["similarity"], 4)
        )
        print("Text:", result["text"])

In [0]:
import importlib
import lakebase

importlib.reload(lakebase)

print(
    hasattr(
        lakebase,
        "search_weather_embeddings",
    )
)

In [0]:
response = test_client.post(
    "/weather/search",
    json={}
)

print(response.status_code)
print(response.get_json())

In [0]:
response = test_client.post(
    "/weather/search",
    json={
        "query": "rain",
        "top_k": "abc"
    }
)

print(response.status_code)
print(response.get_json())